# Day 1 · Lab 2 — SDK Comparison (Stretch)

Same eligibility problem in two stateless SDKs. Compare against Lab 1.

Prerequisite: Lab 1 completed.

## Setup — load env (idempotent)

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv

load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)

# Drop empty-string values that dotenv may have injected
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "":
        del os.environ[k]

assert os.environ.get("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY missing"
print("OK — OPENROUTER_API_KEY loaded")

## Section A — Claude Agent SDK

The Claude Agent SDK reads its config from three env vars. Set them programmatically so we don't depend on `.env` state.

In [ ]:
# Redirect Anthropic SDK to OpenRouter
os.environ["ANTHROPIC_BASE_URL"] = "https://openrouter.ai/api/v1"
os.environ["ANTHROPIC_AUTH_TOKEN"] = os.environ["OPENROUTER_API_KEY"]
os.environ["ANTHROPIC_MODEL"] = "anthropic/claude-sonnet-4.5"
print("ANTHROPIC_BASE_URL:", os.environ["ANTHROPIC_BASE_URL"])
print("ANTHROPIC_MODEL:", os.environ["ANTHROPIC_MODEL"])

### Eligibility check via Claude Agent SDK

Note: OpenRouter routing supports chat completions well; native MCP and sub-agent features are more reliable on the direct Anthropic endpoint. This lab uses simple query() loops which work fine.

In [ ]:
import asyncio
from claude_agent_sdk import query


async def eligibility_via_claude_sdk(application_id: str, income: float, amount: float):
    prompt = (
        f"You are a loan eligibility analyst. Given monthly income {income:.0f} "
        f"and loan amount {amount:.0f} for application {application_id}, "
        f"compute an eligibility score (0-100) based on affordability, "
        f"and recommend approve/review/reject. Reply with a short one-line summary."
    )
    messages = []
    async for msg in query(prompt=prompt):
        messages.append(msg)
    return messages


print("Calling Claude Agent SDK via OpenRouter...")
result = asyncio.run(eligibility_via_claude_sdk("APP-CLAUDE-001", 4500, 200_000))
print(f"\nGot {len(result)} messages. Last 3:")
for m in result[-3:]:
    print(" ", str(m)[:200])

### Claude Agent SDK — takeaways
- ~10 lines for one call
- Stateless — every call independent
- No native HITL — you'd build it yourself
- Fits: simple tool-calling loops, MCP-first flows (best on native endpoint), quick prototypes

## Section B — OpenAI Agents SDK

Explicit `AsyncOpenAI` client. Model ID provider-prefixed.

In [ ]:
from agents import Agent, Runner, AsyncOpenAI, OpenAIChatCompletionsModel
from agents import set_tracing_disabled

set_tracing_disabled(True)   # we use LangSmith, not OpenAI's built-in tracing

client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

eligibility_agent = Agent(
    name="Loan Eligibility Analyst",
    instructions=(
        "You are a loan eligibility analyst. Given an applicant's monthly income "
        "and loan amount, compute an eligibility score (0-100) and recommend "
        "approve/review/reject. Reply with a short one-line summary."
    ),
    model=OpenAIChatCompletionsModel(
        model="openai/gpt-4o",
        openai_client=client,
    ),
)
print("Agent constructed.")

In [ ]:
prompt = "Applicant APP-OAI-001: monthly income 4500, loan amount 200000. Assess eligibility."
result = Runner.run_sync(eligibility_agent, prompt)
print("Final output:")
print(" ", result.final_output)

### OpenAI Agents SDK — takeaways
- ~15 lines of setup
- Stateless
- Native `handoffs` for lightweight multi-agent
- Fits: rapid prototyping, swarm patterns

## Section C — Comparison

| Dimension | LangGraph (Lab 1) | Claude Agent SDK | OpenAI Agents SDK |
|---|---|---|---|
| Setup LOC | ~60 | ~10 | ~15 |
| State model | Typed persistent | Stateless | Stateless |
| Checkpointing | Native (Postgres) | Build manually | Build manually |
| HITL | Native (`interrupt_before`) | Build manually | Build manually |
| Multi-agent | Sub-graphs + supervisor | Via query composition | Via `handoffs` |
| Best for | Enterprise stateful workflows | Basic agent loops | Rapid prototypes |

## Decision heuristic (slide 17)

1. Need HITL? → **LangGraph**
2. State across restarts? → **LangGraph + PostgresSaver**
3. MCP-first, minimal routing? → **Claude Agent SDK** (native endpoint) or **LangGraph**
4. Shared typed multi-agent state? → **LangGraph multi-agent**
5. Rapid prototype < 1 day? → **Claude / OpenAI SDK**

Default: Q1 or Q2 = yes → LangGraph. Otherwise → SDK that matches your tooling.